## Config

In [ ]:
# ============================================================
# CONFIG  — edit only this cell
# ============================================================

PRETRAINED_ADAPTER_PATH = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
DATASET_PATH            = "/kaggle/input/datasets/rohit4118/nemotron/crystal_math_rewritten.csv"
BASE_MODEL_NAME         = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"   # HF slug → written into adapter_config
BASE_MODEL_PATH         = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OFFLINE_PKGS            = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"

# ── Output paths (all derived here — never redefined later) ───────────────────
OUTPUT_DIR             = "/kaggle/working"
GRPO_RUN_DIR           = f"{OUTPUT_DIR}/grpo_output"
ADAPTER_DIR            = f"{OUTPUT_DIR}/grpo_adapter"          # FIX(doc): was inside training cell
SUBMISSION_ADAPTER_DIR = f"{OUTPUT_DIR}/submission_adapter"
SUBMISSION_ZIP         = f"{OUTPUT_DIR}/submission.zip"

# ── Data ──────────────────────────────────────────────────────────────────────
SEED            = 42
GRPO_SAMPLES    = 450
PROMPT_SUFFIX   = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

print({
    "PRETRAINED_ADAPTER_PATH": PRETRAINED_ADAPTER_PATH,
    "DATASET_PATH":            DATASET_PATH,
    "BASE_MODEL_NAME":         BASE_MODEL_NAME,
    "GRPO_SAMPLES":            GRPO_SAMPLES,
    "ADAPTER_DIR":             ADAPTER_DIR,
})


## Triton Installation

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
print("Custom target added:", target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))


## ptxas-Blackwell Patch

In [ ]:
import sys, os, shutil, stat

sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")

ptxas_src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"
ptxas_dst = "/tmp/ptxas-blackwell"

if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    src_bin = os.path.dirname(ptxas_src)
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
    os.environ["TRITON_PTXAS_PATH"]           = ptxas_dst

    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: "12.0"

print("Training environment fixes applied.")


## TRL & Mamba Dependencies

In [ ]:
import glob, importlib.util, os, subprocess, sys, types

def sh(cmd: str, check: bool = True):
    print("+", cmd)
    return subprocess.run(cmd, shell=True, check=check)

def find_spec(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def recursive_wheels(pattern: str):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

import torch
py_tag   = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
abi_tag  = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"

print(f"Python {sys.version} | torch {torch.__version__} | CUDA {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required.")

def pick_best(wheels):
    exact   = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
    if exact:   return exact[-1]
    py_only = [w for w in wheels if py_tag in w]
    return py_only[-1] if py_only else None

if not find_spec("datasets"):
    w = pick_best(recursive_wheels("datasets-*.whl"))
    if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
if not find_spec("trl"):
    w = pick_best(recursive_wheels("trl-*.whl"))
    if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
for pkg, pat in [("multiprocess", "multiprocess-*.whl"),
                 ("dill",         "dill-*.whl"),
                 ("xxhash",       "xxhash-*.whl")]:
    if not find_spec(pkg):
        w = pick_best(recursive_wheels(pat))
        if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"', check=False)

# FIX(fork): direct assignment (not setdefault) — overwrites any stale partial
# submodule from a prior cell run in the same kernel session.
# Must be done BEFORE installing/importing mamba_ssm.
for _mod in ["mamba_ssm.modules.mamba3",
             "mamba_ssm.ops.cute",
             "mamba_ssm.ops.cute.mamba3",
             "mamba_ssm.ops.cute.mamba3.mamba3_step_fn"]:
    sys.modules[_mod] = types.ModuleType(_mod)
sys.modules["mamba_ssm.modules.mamba3"].Mamba3 = None

if not find_spec("mamba_ssm"):
    causal_w = pick_best(recursive_wheels("causal*conv1d*.whl"))
    mamba_w  = pick_best(recursive_wheels("mamba_ssm-*.whl"))
    if causal_w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{causal_w}"')
    if mamba_w:  sh(f'{sys.executable} -m pip install --no-index --no-deps "{mamba_w}"')

# Install trl if needed (for training mode)
if True:
    try:
        import trl
        print(f"trl already installed: {trl.__version__}")
    except ImportError:
        # Try offline install first, then online
        offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
        if os.path.exists(offline_path):
            subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
        else:
            subprocess.run("pip install trl", shell=True)
        import trl
        print(f"trl installed: {trl.__version__}")


import datasets, trl, mamba_ssm
print(f"datasets: {datasets.__version__}  trl: {trl.__version__}  mamba_ssm: {mamba_ssm.__version__}")
print("✓ Dependencies ready")


## Load Base Model + Pre-trained LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# FIX: Use explicit "cuda:0" instead of "auto". 
# "auto" can sometimes fallback to CPU or fragment devices, triggering 
# the "input_ids on cuda, model on cpu" mismatch during GRPO generation.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    device_map="cuda:0", 
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print("Base model loaded.")

model = PeftModel.from_pretrained(
    base_model,
    PRETRAINED_ADAPTER_PATH,
    is_trainable=True,
)

# Explicitly move the entire model (base + LoRA adapters) to the GPU 
# to guarantee device alignment before training begins.
model = model.to("cuda")

# Quick sanity check
assert next(model.parameters()).is_cuda, "⚠️ Model is still on CPU! Check GPU availability."

model.print_trainable_parameters()
print("Pre-trained LoRA loaded and ready for GRPO.")

## Data Preparation (100 Rows)

In [ ]:
import pandas as pd
from datasets import Dataset as HFDataset

# Load data
df = pd.read_csv(DATASET_PATH)

# The code previously executed successfully with these column names, so we keep them for reading.
# Note: If your CSV actually uses 'prompt' instead of 'problem', change row["problem"] to row["prompt"].
df = df.dropna(subset=["solution", "problem", "answer"])
df = df.sample(n=min(GRPO_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print(f"Loaded {len(df)} rows.")

# FIX: TRL GRPO requires the prompt column to be named "prompt", not "problem".
records = []
for _, row in df.iterrows():
    records.append({
        "prompt": str(row["problem"]) + PROMPT_SUFFIX,  # Changed key from "problem" to "prompt"
        "answer": str(row["answer"]),
    })

dataset = HFDataset.from_list(records)
print(f"Dataset ready — {len(dataset)} records")
print(f"  Sample prompt [:100]: {records[0]['prompt'][:100]}")
print(f"  Sample answer       : {records[0]['answer']}")

## Reward Functions

In [ ]:
import re

# ── extract_response (from document) ──────────────────────────────────────────
# Defensive helper: TRL normally passes plain strings but some versions pass
# list-of-dicts [{role, content}]. This handles both safely.
def extract_response(comp) -> str:
    if isinstance(comp, list):
        return comp[-1]["content"] if comp else ""
    return str(comp)

# ── correctness_reward_func ───────────────────────────────────────────────────
# FIX(doc):  removed `prompts` as first positional arg — TRL calls
#            reward_func(completions=..., **dataset_cols) without passing
#            `prompts` positionally → TypeError: missing required argument.
# FIX(fork): `answer` is a LIST (TRL broadcasts it once per completion).
#            str(["42","42"]) = "['42','42']" → never matches → reward = 0.
#            Fixed by iterating zip(completions, answers).
def correctness_reward_func(completions, answer, **kwargs):
    """
    +1.0 if the last \boxed{} in the completion matches ground truth, else 0.0.
    `answer` is a list — one ground-truth string per completion.
    """
    rewards  = []
    # Guard: accept both list (TRL ≥ 0.12) and scalar (older TRL)
    answers  = answer if isinstance(answer, (list, tuple)) else [answer] * len(completions)
    for comp, gt in zip(completions, answers):
        text    = extract_response(comp)
        matches = re.findall(r"\\boxed\{([^}]*)\}", text)
        pred    = matches[-1].strip() if matches else ""
        rewards.append(1.0 if pred == str(gt).strip() else 0.0)
    return rewards

# ── format_reward_func ────────────────────────────────────────────────────────
# FIX(doc): removed `prompts` positional arg (same reason as above)
def format_reward_func(completions, **kwargs):
    """
    +0.2 if <think>…</think> present (chain-of-thought structure)
    +0.1 if \boxed{} present
    """
    rewards = []
    for comp in completions:
        text = extract_response(comp)
        r    = 0.0
        if "<think>" in text and "</think>" in text:
            r += 0.2
        if re.search(r"\\boxed\{", text):
            r += 0.1
        rewards.append(r)
    return rewards

print("Reward functions ready.")
print("  correctness_reward_func : +1.0 exact boxed match, 0.0 otherwise")
print("  format_reward_func      : +0.2 think tags, +0.1 boxed present")


## GRPO Training

In [ ]:
import time, inspect
from trl import GRPOConfig, GRPOTrainer

# ── Detect valid GRPOConfig param names at runtime ───────────────────────────
# The generation-length arg was renamed across TRL versions:
#   TRL <= 0.11 : max_completion_length
#   TRL 0.12-0.14: max_new_tokens
#   TRL >= 0.15 : max_completion_length  (reverted)
# temperature / use_vllm also moved in/out of GRPOConfig across versions.
# Hard-coding either name breaks on a different TRL version — inspect instead.
_valid = set(inspect.signature(GRPOConfig.__init__).parameters)
print(f"TRL version : {__import__('trl').__version__}")
print(f"GRPOConfig generation-related params: "
      f"{sorted(p for p in _valid if any(k in p for k in ('max','temp','gen','token','complet','vllm')))}")

def _pick(candidates, value):
    """Return {first_matching_param_name: value}, or {} if none found."""
    for name in candidates:
        if name in _valid:
            return {name: value}
    return {}

_gen_len = _pick(["max_completion_length", "max_new_tokens"], 512)
_prompt_len = _pick(["max_prompt_length"],512)
_temp    = _pick(["temperature"], 0.9)
_vllm    = _pick(["use_vllm"], False)
print(f"  gen-length kwarg : {_gen_len}")
print(f"  temperature kwarg: {_temp}")
print(f"  use_vllm kwarg   : {_vllm}")

# ── GRPOConfig ────────────────────────────────────────────────────────────────
training_args = GRPOConfig(
    output_dir                    = GRPO_RUN_DIR,
    num_train_epochs              = 1,
    per_device_train_batch_size   = 4,
    gradient_accumulation_steps   = 2,
    learning_rate                 = 5e-6,
    lr_scheduler_type             = "cosine",
    warmup_ratio                  = 0.1,
    #max_completion_length         = 512,
    num_generations               = 2,
    logging_steps                 = 10,
    save_strategy                 = "no",
    bf16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    dataloader_num_workers        = 4,
    remove_unused_columns         = False,
    seed                          = SEED,
    report_to                     = "none",
    **_gen_len,   # max_completion_length OR max_new_tokens — whichever this TRL accepts
    **_temp,      # temperature — present in some TRL versions only
    **_vllm,      # use_vllm   — present in some TRL versions only
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [correctness_reward_func, format_reward_func],
    args             = training_args,
    train_dataset    = dataset,
)

print("\nStarting GRPO training…")
print(f"  Dataset size    : {len(dataset)}")
print(f"  Effective batch : {training_args.per_device_train_batch_size} × "
      f"{training_args.gradient_accumulation_steps} × "
      f"{training_args.num_generations} = "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * training_args.num_generations}")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\nGRPO done in {elapsed/60:.1f} min")


## Save Adapter

In [ ]:
import os

os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

for fname in ["adapter_config.json", "adapter_model.safetensors"]:
    fpath = os.path.join(ADAPTER_DIR, fname)
    if os.path.exists(fpath):
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
    else:
        print(f"  WARNING: {fname} not found!")


## Package submission.zip

In [ ]:
import json, os, shutil, zipfile

os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]
print(f"Packaging adapter from: {ADAPTER_DIR}")

for fname in required_files:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"  Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME   # HF slug
cfg["inference_mode"]           = True
cfg["lora_dropout"]             = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_ADAPTER_DIR, fname), fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB  →  {SUBMISSION_ZIP}")
print("Done! Ready to submit.")


## Verification

In [ ]:
import os

checks = {
    "Adapter input path exists"       : os.path.exists(PRETRAINED_ADAPTER_PATH),
    "GRPO samples = 100"              : len(dataset) == 100,
    "adapter_config.json saved"       : os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")),
    "adapter_model.safetensors saved" : os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")),
    "submission.zip created"          : os.path.exists(SUBMISSION_ZIP),
    "inference_mode = True"           : cfg.get("inference_mode") == True,
    "lora_dropout = 0.0"              : cfg.get("lora_dropout") == 0.0,
    "base_model is HF slug"           : cfg.get("base_model_name_or_path") == BASE_MODEL_NAME,
}

print("\n" + "="*55)
print("VERIFICATION SUMMARY")
print("="*55)
all_pass = True
for check, passed in checks.items():
    if not passed: all_pass = False
    print(f"  {'✓ PASS' if passed else '✗ FAIL'}: {check}")
print("="*55)
print("All checks PASSED ✓" if all_pass else "Some checks FAILED — see above ✗")
